# Webcam word detection (custom words)

This notebook runs the **custom-words** isolated-word lip-reading model in `checkpoints/custom_words.pt` against a live webcam clip. The model recognises exactly these 8 words: **hello, hi, yes, no, good, bad, thanks, bye**.

Say **one** of those words, then press **q** to stop recording and classify the captured mouth frames.

In [ ]:
from pathlib import Path
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from models.LBY.model import LipReadingClassifier
from processing.preprocessor import Preprocessor
from processing.video_loader import VideoLoader


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'checkpoints' / 'custom_words.pt').is_file() and (candidate / 'models').is_dir():
            return candidate
    raise FileNotFoundError(
        'Could not find the VisionSpeak project root containing checkpoints/custom_words.pt.'
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')

In [ ]:
CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'custom_words.pt'

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
label_to_index = checkpoint['label_to_index']
index_to_label = {index: word for word, index in label_to_index.items()}

model = LipReadingClassifier(num_classes=len(label_to_index)).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Loaded checkpoint: {CHECKPOINT_PATH.name} (epoch {checkpoint["epoch"]})')
print(f'Vocabulary ({len(index_to_label)} words): {sorted(index_to_label.values())}')

In [ ]:
def prepare_custom_frames(frames: np.ndarray) -> torch.Tensor:
    """Match training preprocessing: /255 then per-clip standardisation."""
    tensor = torch.from_numpy(np.asarray(frames, dtype=np.float32)) / 255.0
    tensor = (tensor - tensor.mean()) / (tensor.std() + 1e-6)
    return tensor.unsqueeze(1).unsqueeze(0)  # (1, T, 1, H, W)

## Record one word

Run the next cell. An OpenCV preview opens with the full webcam view and a green box around the mouth crop. Say **one** of the 8 words, then press **q**. The window closes automatically.

In [ ]:
captured_frames = None
video_data = None

try:
    print('Say one word, then press q to stop and predict.')
    video_data = VideoLoader().load('webcam', show_full_frame=True)
    if not video_data.get('loaded'):
        raise ValueError('No webcam frames were captured. Please try again.')

    video_data = Preprocessor().convert_to_greyscale(video_data)
    captured_frames = np.asarray(video_data['frames'], dtype=np.uint8)
    print(f'Captured {len(captured_frames)} mouth-crop frames.')
finally:
    if video_data is not None:
        capture = video_data.get('capture')
        if capture is not None:
            capture.release()
    cv2.destroyAllWindows()

## Predict the word

Reports the winner and the three most likely words.

In [ ]:
if captured_frames is None:
    raise RuntimeError('Run the webcam capture cell before running prediction.')

model_input = prepare_custom_frames(captured_frames).to(device)
with torch.inference_mode():
    probabilities = torch.softmax(model(model_input), dim=1).squeeze(0).cpu()

top_count = min(3, len(index_to_label))
top_probabilities, top_indices = torch.topk(probabilities, k=top_count)
top_predictions = [
    (index_to_label[int(index)], float(probability))
    for probability, index in zip(top_probabilities, top_indices)
]
predicted_word, confidence = top_predictions[0]

display(Markdown(f'## Prediction: **{predicted_word}** ({confidence:.1%})'))
for rank, (word, probability) in enumerate(top_predictions, start=1):
    print(f'{rank}. {word}: {probability:.1%}')

In [ ]:
if captured_frames is None:
    raise RuntimeError('Run the webcam capture cell before inspecting frames.')

plt.figure(figsize=(6, 3))
plt.imshow(captured_frames[-1], cmap='gray', vmin=0, vmax=255)
plt.title('Last captured grayscale mouth crop')
plt.axis('off')
plt.show()

print('For best results, record only one word with minimal silence before or after it.')

## Notes

- This model recognises only the 8 custom words: **hello, hi, yes, no, good, bad, thanks, bye**.
- The 71.9% accuracy was measured on a stratified split that mixes the two recording speakers, so live accuracy (especially across a new speaker) may be lower.
- The notebook does not save video. Re-run the capture, prediction, and inspection cells for another attempt.
- If no preview opens, make sure the Jupyter kernel has camera permission and is running in a local GUI-capable Python environment.